# π0.5 SpiritAI 模型推理验证

这个 notebook 可以直接加载 SpiritAI 的 JAX/Orbax checkpoint 或 PyTorch safetensors checkpoint。

加载规则和 `scripts/serve_policy.py` 一致：

- checkpoint 目录下存在 `model.safetensors`：走 PyTorch 推理路径。
- checkpoint 目录下存在 `params/`：走 JAX/Orbax 推理路径。

只需要修改 **参数配置区** 的 `WEIGHT_DIR`、`CONFIG_NAME` 和 `TASK_PROMPT`。


In [1]:
from pathlib import Path

# ==== 参数配置区 ====
# JAX/Orbax checkpoint 示例：目录内包含 params/。
# PyTorch checkpoint 示例：目录内包含 model.safetensors。
WEIGHT_DIR = "checkpoints/pi05_spiritai_lora/20260512_FoldPaperBox_350ep_34000stp/15000"
CONFIG_NAME = "pi05_spiritai_lora"
TASK_PROMPT = "Assemble the cardboard box by erecting the flat sheet and folding the side flaps"

# "fake": 使用 make_spiritai_example() 生成随机数据。
# 自定义观测时，CUSTOM_OBS 需要包含 cam_high/cam_left_wrist/cam_right_wrist/各 state key/prompt。
DATA_MODE = "fake"
CUSTOM_OBS = None

BENCHMARK_ITERS = 5  # warmup 后压测次数，0 跳过


In [2]:
import importlib.util
import os
import time

import numpy as np
from openpi.policies import spiritai_policy
from openpi.policies import policy_config as _policy_config
from openpi.training import config as _config

# Resolve checkpoint path relative to repo root (works regardless of notebook cwd).
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / "pyproject.toml").exists():
    _repo_root = _repo_root.parent

checkpoint_path = Path(WEIGHT_DIR)
if not checkpoint_path.is_absolute():
    checkpoint_path = (_repo_root / checkpoint_path).resolve()
if not checkpoint_path.exists():
    raise FileNotFoundError(f"WEIGHT_DIR not found: {checkpoint_path}")

has_pytorch_weights = (checkpoint_path / "model.safetensors").exists()
has_jax_params = (checkpoint_path / "params").exists()
if has_pytorch_weights and has_jax_params:
    print("[WARN] checkpoint contains both model.safetensors and params/; create_trained_policy will prefer PyTorch.")
elif not has_pytorch_weights and not has_jax_params:
    raise FileNotFoundError(
        f"Checkpoint must contain either model.safetensors or params/: {checkpoint_path}"
    )

BACKEND = "pytorch" if has_pytorch_weights else "jax"

try:
    import psutil
    avail_gb = psutil.virtual_memory().available / 1e9
except ImportError:
    avail_gb = None

print(f"Repo root   : {_repo_root}")
print(f"Checkpoint  : {checkpoint_path}")
print(f"Backend     : {BACKEND}")
if avail_gb is not None:
    print(f"Available RAM: {avail_gb:.1f} GB")

if BACKEND == "pytorch":
    if importlib.util.find_spec("torch") is None:
        raise ImportError("PyTorch checkpoint selected, but torch is not installed.")
    import torch

    weight_bytes = (checkpoint_path / "model.safetensors").stat().st_size
    need_gb = weight_bytes * 2 / 1e9  # rough: need about 2x weight size for loading
    print(f"Weights     : {weight_bytes / 1e9:.1f} GB (need ~{need_gb:.0f} GB RAM for loading)")
    if avail_gb is not None and avail_gb < need_gb:
        print(f"[WARN] Available RAM may be insufficient: {avail_gb:.1f} GB < {need_gb:.0f} GB")
    if torch.cuda.is_available():
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU VRAM    : {vram_gb:.1f} GB ({torch.cuda.get_device_name(0)})")
    else:
        print("GPU VRAM    : N/A (PyTorch will use CPU, inference will be slow)")
else:
    if importlib.util.find_spec("jax") is None:
        raise ImportError("JAX checkpoint selected, but jax is not installed.")
    import jax

    print("JAX devices :", jax.devices())


Repo root   : /home/dengkevin/Documents/code/openpi
Checkpoint  : /home/dengkevin/Documents/code/openpi/checkpoints/pi05_spiritai_lora/20260512_FoldPaperBox_350ep_34000stp/15000
Backend     : jax
Available RAM: 122.4 GB
JAX devices : [CudaDevice(id=0)]


## 构建观测数据

当前 SpiritAI transform 只使用 3 路相机：`cam_high`、`cam_left_wrist`、`cam_right_wrist`。


In [3]:
if DATA_MODE == "fake":
    observation = spiritai_policy.make_spiritai_example()
    observation["prompt"] = TASK_PROMPT
else:
    if CUSTOM_OBS is None:
        raise ValueError("CUSTOM_OBS must be provided when DATA_MODE != 'fake'.")
    observation = dict(CUSTOM_OBS)
    observation.setdefault("prompt", TASK_PROMPT)

required_keys = [
    "cam_high",
    "cam_left_wrist",
    "cam_right_wrist",
    *spiritai_policy.STATE_KEYS,
]
missing = [k for k in required_keys if k not in observation]
if missing:
    raise KeyError(f"Observation is missing required keys: {missing}")

state_dim = sum(np.asarray(observation[k]).flatten().shape[0] for k in spiritai_policy.STATE_KEYS)
print(f"state dim: {state_dim} | prompt: {observation['prompt']}")
for k in ("cam_high", "cam_left_wrist", "cam_right_wrist"):
    arr = np.asarray(observation[k])
    print(f"  {k}: {arr.shape} {arr.dtype}")


state dim: 27 | prompt: Assemble the cardboard box by erecting the flat sheet and folding the side flaps
  cam_high: (480, 640, 3) uint8
  cam_left_wrist: (480, 640, 3) uint8
  cam_right_wrist: (480, 640, 3) uint8


## 加载权重并推理


In [4]:
config = _config.get_config(CONFIG_NAME)
print(
    f"Config: {config.name} | pi05={config.model.pi05} | "
    f"action_dim={config.model.action_dim} | action_horizon={config.model.action_horizon}"
)

sample_kwargs = None
pytorch_device = None
if BACKEND == "pytorch":
    import torch
    pytorch_device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"PyTorch device: {pytorch_device}")

try:
    t0 = time.time()
    policy = _policy_config.create_trained_policy(
        config,
        str(checkpoint_path),
        sample_kwargs=sample_kwargs,
        pytorch_device=pytorch_device,
    )
    print(f"Policy loaded ({time.time() - t0:.1f}s) | pytorch={policy._is_pytorch_model}")
except RuntimeError as e:
    if BACKEND == "pytorch" and ("out of memory" in str(e).lower() or "CUDA" in str(e)):
        print(f"[ERROR] CUDA OOM: {e}")
        print("Trying PyTorch CPU fallback ...")
        torch.cuda.empty_cache()
        t0 = time.time()
        policy = _policy_config.create_trained_policy(
            config,
            str(checkpoint_path),
            sample_kwargs=sample_kwargs,
            pytorch_device="cpu",
        )
        print(f"Policy loaded on CPU ({time.time() - t0:.1f}s) | pytorch={policy._is_pytorch_model}")
    else:
        raise

# First inference includes compilation overhead for JAX.
t0 = time.time()
result = policy.infer(observation)
print(f"Inference: {((time.time() - t0) * 1000):.0f} ms")

actions = result["actions"]
expected = (config.model.action_horizon, spiritai_policy.ACTION_DIM)
assert actions.shape == expected, f"shape {actions.shape} != expected {expected}"
print(f"Actions shape: {actions.shape} | dtype: {actions.dtype} | range: [{actions.min():.4f}, {actions.max():.4f}]")

if BENCHMARK_ITERS > 0:
    for _ in range(3):
        policy.infer(observation)
    times = []
    for _ in range(BENCHMARK_ITERS):
        t0 = time.time()
        policy.infer(observation)
        times.append((time.time() - t0) * 1000)
    print(f"Benchmark ({BENCHMARK_ITERS} iters): mean={np.mean(times):.0f}ms std={np.std(times):.0f}ms")

del policy
print("Done.")


Config: pi05_spiritai_lora | pi05=True | action_dim=32 | action_horizon=10
Policy loaded (4.8s) | pytorch=False
Inference: 16755 ms
Actions shape: (10, 27) | dtype: float64 | range: [-1.7434, 5.1305]
Benchmark (5 iters): mean=152ms std=0ms
Done.
